# 3 - Social Media Ad Conversion Prediction - Classification

<img src='https://images.unsplash.com/photo-1460925895917-afdab827c52f?auto=format&fit=crop&w=1200&q=80'>

Bu çalışmada dijital pazarlama kampanya verilerini kullanarak bir müşterinin dönüşüm yapıp yapmayacağını tahmin eden bir classification modeli geliştireceğim. Bu proje özellikle sosyal medya reklamları ve dijital kampanya performansını anlamak için kullanılabilir.

## Akış

1. Veriyi yükleme
2. Veriyi okuma ve inceleme
3. Veri temizleme
4. Feature engineering
5. Train-test split
6. Classification modelleri kurma
7. En iyi modeli değerlendirme
8. Örnek tahmin
9. Sonuç

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

## 1. Veriyi Yükleme

In [ ]:
# Bu projede Kaggle'dan indirilen dijital pazarlama datasetini Colab üzerinden zip dosyası olarak açıp kullanacağım.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/Predict Conversion in Digital Marketing Dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content')

os.listdir('/content')[:20]

## 2. Veriyi Okuma ve İnceleme

In [ ]:
# Bu bölümde zip içinden çıkan csv dosyasını okuyup veri setinin yapısını inceleyeceğim.

In [ ]:
file_path = '/content/digital_marketing_campaign_dataset.csv'

df = pd.read_csv(file_path)
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Veri Temizleme

In [ ]:
# Bu bölümde boş veri kontrolü yapacağım ve dönüşüm dağılımını inceleyeceğim.

In [ ]:
df.isnull().sum()

In [ ]:
df['Conversion'].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Conversion', data=df)
plt.title('Conversion Distribution')
plt.show()

## 4. Feature Engineering

In [ ]:
# Bu bölümde CustomerID sütununu çıkarıp kategorik sütunları sayısal hale getireceğim.

In [ ]:
df = df.drop('CustomerID', axis=1)
df = pd.get_dummies(df, drop_first=True)
df.head()

## 5. Train-Test Split

In [ ]:
# Bu bölümde dönüşüm sütununu hedef değişken olarak ayırıp veriyi eğitim ve test olarak böleceğim.

In [ ]:
x = df.drop('Conversion', axis=1)
y = df['Conversion']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

x_train.shape, x_test.shape

## 6. Classification Modelleri Kurma

In [ ]:
# Bu bölümde birkaç farklı classification modeli kurup sonuçları karşılaştıracağım.

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier())
    ]),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append([name, accuracy, f1])
    trained_models[name] = model

results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'F1'])
results_df.sort_values(by='F1', ascending=False)

## 7. En İyi Modeli Değerlendirme

In [ ]:
# Bu bölümde en başarılı modeli seçip detaylı sonuçlarını inceleyeceğim.

In [ ]:
best_model_name = results_df.sort_values(by='F1', ascending=False).iloc[0]['Model']
best_model = trained_models[best_model_name]
best_pred = best_model.predict(x_test)

print('Best Model:', best_model_name)
print('Accuracy:', accuracy_score(y_test, best_pred))
print('F1 Score:', f1_score(y_test, best_pred))
print(classification_report(y_test, best_pred))

In [ ]:
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 8. Örnek Tahmin

In [ ]:
# Bu bölümde örnek bir müşteri verisi için dönüşüm tahmini yapacağım.

In [ ]:
sample_customer = x_test.iloc[[0]]
sample_prediction = best_model.predict(sample_customer)[0]

print('Tahmin edilen dönüşüm durumu:', sample_prediction)
print('Gerçek değer:', y_test.iloc[0])

## 9. Sonuç

In [ ]:
# Bu bölümde en iyi modeli yorumlayıp genel sonucu özetleyeceğim.

In [ ]:
best_accuracy = results_df.sort_values(by='F1', ascending=False).iloc[0]['Accuracy']
best_f1 = results_df.sort_values(by='F1', ascending=False).iloc[0]['F1']

print(f"Bu projede en başarılı model: {best_model_name}")
print(f"Accuracy: {best_accuracy:.4f}")
print(f"F1 Score: {best_f1:.4f}")
print('Bu sonuca göre dijital pazarlama dönüşüm tahmininde en iyi performansı bu model verdi.')